In [3]:



import numpy as np
import os
import h5py

SAMPLES = 4000
SEED = 14

#hide tensorflow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

#use cpu
# os.environ["CUDA_VISIBLE_DEVICES"] = "-1"



In [4]:

#Path to directory
directory = r"C:\Users\Alex\Music\ComputingTechnologyProjectBP45\benchmark\content\data"

# List HDF5 file names
h5_files = [
    os.path.join(directory, file_name)
    for file_name in os.listdir(directory)
    if file_name.endswith('.h5')
]

np.random.seed(SEED)
np.random.shuffle(h5_files)

# Limit the in-memory dataset to keep training within available RAM.
selected_files = h5_files[:SAMPLES]

# Split the dataset into train and validation sets (80:20).
split_idx = int(0.8 * len(selected_files))
val_files = selected_files[split_idx:]


def load_h5_arrays(files):
    images = []
    masks = []
    for file_path in files:
        with h5py.File(file_path, 'r') as h5_file:
            images.append(np.asarray(h5_file['image'], dtype=np.float32))
            masks.append(np.asarray(h5_file['mask'], dtype=np.float32))
    return np.stack(images), np.stack(masks)



X_test, Y_test = load_h5_arrays(val_files)


print("Test data shape:", X_test.shape)
print("Test data dtype:", X_test.dtype)

X_test = np.asarray(X_test, dtype=np.float32)


Test data shape: (800, 240, 240, 4)
Test data dtype: float32


Show slices

Predictions for python 3.12

In [8]:
import numpy as np
import keras
import tensorflow as tf

print("NumPy version:", np.__version__)
print("Keras version:", keras.__version__)
print("TensorFlow version:", tf.__version__)

NumPy version: 1.26.4
Keras version: 3.3.3
TensorFlow version: 2.16.1


In [ ]:
import os
import shutil
import tempfile
import h5py
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf


def dice_coef(y_true, y_pred):
    smooth = 1e-6

    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    y_true = tf.reshape(y_true, [-1])
    y_pred = tf.reshape(y_pred, [-1])

    intersection = tf.reduce_sum(y_true * y_pred)

    dice = (
        (2.0 * intersection + smooth)
        / (
            tf.reduce_sum(y_true)
            + tf.reduce_sum(y_pred)
            + smooth
        )
    )

    return dice


# Custom binary Dice metric
def binary_dice_coef(y_true, y_pred):
    y_pred = tf.cast(y_pred >= 0.5, tf.float32)
    return dice_coef(y_true, y_pred)


# Custom Dice loss
def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)


# Combined BCE + Dice loss
def combined_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)

    return tf.reduce_mean(bce) + dice_loss(y_true, y_pred)


# Load the saved CNN model and handle legacy HDF5 files named with a .keras suffix.
model_path = os.path.join(
    os.path.dirname(os.path.abspath("cnn_semantic.ipynb")),
    "unet_brain_model_best.keras"
)

if not os.path.isfile(model_path):
    raise FileNotFoundError(f"Model file not found: {model_path}")

custom_objects = {
    "dice_coef": dice_coef,
    "binary_dice_coef": binary_dice_coef,
    "dice_loss": dice_loss,
    "combined_loss": combined_loss,
}

if h5py.is_hdf5(model_path):
    class CompatibleConv2DTranspose(tf.keras.layers.Conv2DTranspose):
        @classmethod
        def from_config(cls, config):
            config = dict(config)
            groups = config.pop("groups", 1)
            if groups != 1:
                raise ValueError(
                    f"Cannot load Conv2DTranspose with groups={groups}"
                )
            return super().from_config(config)

    with tempfile.TemporaryDirectory() as temp_dir:
        legacy_model_path = os.path.join(
            temp_dir,
            "unet_brain_model_best.h5",
        )
        shutil.copy2(model_path, legacy_model_path)
        brain_model = tf.keras.models.load_model(
            legacy_model_path,
            custom_objects={
                **custom_objects,
                "Conv2DTranspose": CompatibleConv2DTranspose,
            },
            compile=False,
        )
else:
    brain_model = tf.keras.models.load_model(
        model_path,
        custom_objects=custom_objects,
        compile=False,
    )

print("Model input shape:", brain_model.input_shape)
print("Test data shape:", X_test.shape)

y_pred = brain_model.predict(X_test, verbose=0)


# Randomly select distinct test images with non-empty ground-truth masks.
valid_indices = np.flatnonzero(
    np.any(Y_test > 0, axis=(1, 2, 3))
)

if len(valid_indices) == 0:
    raise ValueError("No test images contain a ground-truth mask.")

number_of_examples = min(100, len(valid_indices))

selected_indices = np.random.choice(
    valid_indices,
    size=number_of_examples,
    replace=False,
)

print("Random test images with ground truth:", selected_indices)


plt.style.use("ggplot")
plt.rcParams["figure.facecolor"] = "#171717"
plt.rcParams["text.color"] = "#AAAAAA"


def display_mask_channels_as_rgb(
    mask,
    title="Mask Channels as RGB",
):
    channel_names = [
        "Necrotic (NEC)",
        "Edema (ED)",
        "Tumour (ET)",
    ]

    fig, axes = plt.subplots(1, 3, figsize=(9.75, 5))

    for idx, ax in enumerate(axes):
        rgb_mask = np.zeros(
            (mask.shape[1], mask.shape[2], 3),
            dtype=np.uint8,
        )

        rgb_mask[..., idx] = (
            mask[idx, :, :] * 255
        ).astype(np.uint8)

        ax.imshow(rgb_mask)
        ax.axis("off")
        ax.set_title(channel_names[idx])

    plt.suptitle(title, fontsize=20, y=0.93)
    plt.tight_layout()
    plt.show()


dice_scores = []

for example_number, idx in enumerate(selected_indices, start=1):
    ground_truth = Y_test[idx]
    prediction = y_pred[idx]

    intersection = np.sum(ground_truth * prediction)

    dice_score = (
        (2.0 * intersection + 1e-6)
        / (
            np.sum(ground_truth)
            + np.sum(prediction)
            + 1e-6
        )
    )

    dice_scores.append(dice_score)

    print(
        f"Example {example_number}: image index {idx}, "
        f"Dice score {dice_score:.4f}"
    )

    display_mask_channels_as_rgb(
        ground_truth.transpose(2, 0, 1),
        title=f"Actual Mask - Example {example_number}",
    )

    display_mask_channels_as_rgb(
        prediction.transpose(2, 0, 1),
        title=(
            f"Predicted Model Mask - Example {example_number} "
            f"(Dice: {dice_score:.4f})"
        ),
    )


print(f"\nAverage Dice score: {np.mean(dice_scores):.4f}")



In [ ]:
import os
import shutil
import tempfile
import h5py
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf


# ============================================================
# Configuration
# ============================================================

directory = (
    r"C:\Users\Alex\Music\ComputingTechnologyProjectBP45"
    r"\benchmark\content\data"
)

model_path = os.path.join(
    os.path.dirname(os.path.abspath("cnn_semantic.ipynb")),
    "unet_brain_model_best.keras",
)

target_filename = "volume_1_slice_37.h5"
image_path = os.path.join(directory, target_filename)


# ============================================================
# Custom metrics and losses
# ============================================================

def dice_coef(y_true, y_pred):
    smooth = 1e-6

    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    y_true = tf.reshape(y_true, [-1])
    y_pred = tf.reshape(y_pred, [-1])

    intersection = tf.reduce_sum(y_true * y_pred)

    return (
        (2.0 * intersection + smooth)
        / (
            tf.reduce_sum(y_true)
            + tf.reduce_sum(y_pred)
            + smooth
        )
    )


def binary_dice_coef(y_true, y_pred):
    y_pred = tf.cast(y_pred >= 0.5, tf.float32)
    return dice_coef(y_true, y_pred)


def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)


def combined_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(
        y_true,
        y_pred,
    )
    return tf.reduce_mean(bce) + dice_loss(y_true, y_pred)


custom_objects = {
    "dice_coef": dice_coef,
    "binary_dice_coef": binary_dice_coef,
    "dice_loss": dice_loss,
    "combined_loss": combined_loss,
}


# ============================================================
# Load saved model
# ============================================================

if not os.path.isfile(model_path):
    raise FileNotFoundError(
        f"Model file not found: {model_path}"
    )

if h5py.is_hdf5(model_path):

    class CompatibleConv2DTranspose(
        tf.keras.layers.Conv2DTranspose
    ):
        @classmethod
        def from_config(cls, config):
            config = dict(config)
            groups = config.pop("groups", 1)

            if groups != 1:
                raise ValueError(
                    f"Cannot load Conv2DTranspose with groups={groups}"
                )

            return super().from_config(config)

    with tempfile.TemporaryDirectory() as temp_dir:
        legacy_model_path = os.path.join(
            temp_dir,
            "unet_brain_model_best.h5",
        )

        shutil.copy2(model_path, legacy_model_path)

        brain_model = tf.keras.models.load_model(
            legacy_model_path,
            custom_objects={
                **custom_objects,
                "Conv2DTranspose": CompatibleConv2DTranspose,
            },
            compile=False,
        )

else:
    brain_model = tf.keras.models.load_model(
        model_path,
        custom_objects=custom_objects,
        compile=False,
    )

print("Model loaded successfully.")
print("Model input shape:", brain_model.input_shape)
print("Model output shape:", brain_model.output_shape)


# ============================================================
# Load one specific MRI slice
# ============================================================

if not os.path.isfile(image_path):
    raise FileNotFoundError(
        f"MRI file not found: {image_path}"
    )

with h5py.File(image_path, "r") as h5_file:
    image = np.asarray(
        h5_file["image"],
        dtype=np.float32,
    )

    ground_truth = np.asarray(
        h5_file["mask"],
        dtype=np.float32,
    )

print("\nMRI file:", target_filename)
print("Original image shape:", image.shape)
print("Ground-truth mask shape:", ground_truth.shape)


# ============================================================
# Preprocess MRI image
# Must match the preprocessing used during model training
# ============================================================

def preprocess_mri_image(image):
    image = np.asarray(image, dtype=np.float32)

    if image.ndim != 3:
        raise ValueError(
            f"Expected image shape H x W x C, got {image.shape}"
        )

    image = np.nan_to_num(
        image,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    brain_mask = np.any(image != 0, axis=-1)

    processed = np.zeros_like(
        image,
        dtype=np.float32,
    )

    for channel in range(image.shape[-1]):
        channel_data = image[..., channel]
        brain_voxels = channel_data[brain_mask]

        if brain_voxels.size == 0:
            continue

        lower = np.percentile(brain_voxels, 0.5)
        upper = np.percentile(brain_voxels, 99.5)

        clipped = np.clip(
            channel_data,
            lower,
            upper,
        )

        valid_voxels = clipped[brain_mask]

        mean = np.mean(valid_voxels)
        std = np.std(valid_voxels)

        if std > 1e-6:
            normalised = (clipped - mean) / std
        else:
            normalised = clipped - mean

        processed[..., channel] = np.where(
            brain_mask,
            normalised,
            0.0,
        )

    return processed.astype(np.float32)


image = preprocess_mri_image(image)

# Add batch dimension: (H, W, C) -> (1, H, W, C)
X_single = np.expand_dims(image, axis=0)

print("Preprocessed input shape:", X_single.shape)


# ============================================================
# Predict tumour segmentation
# ============================================================

prediction = brain_model.predict(
    X_single,
    verbose=0,
)[0]

print("Prediction shape:", prediction.shape)


# ============================================================
# Prepare ground-truth and predicted masks
# ============================================================

if ground_truth.shape != prediction.shape:
    raise ValueError(
        "Ground-truth and prediction shapes do not match: "
        f"{ground_truth.shape} vs {prediction.shape}"
    )

# Continuous probabilities
predicted_probabilities = np.clip(
    prediction,
    0.0,
    1.0,
)

# Binary masks using threshold 0.5
predicted_binary = (
    predicted_probabilities >= 0.5
).astype(np.float32)

ground_truth_binary = (
    ground_truth >= 0.5
).astype(np.float32)


# ============================================================
# Calculate Dice scores
# ============================================================

def calculate_dice(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)

    intersection = np.sum(y_true * y_pred)

    return float(
        (2.0 * intersection + 1e-6)
        / (
            np.sum(y_true)
            + np.sum(y_pred)
            + 1e-6
        )
    )


channel_names = [
    "NEC",
    "ED",
    "ET",
]

print("\nDice scores at threshold 0.5:")

for channel, name in enumerate(channel_names):
    score = calculate_dice(
        ground_truth_binary[..., channel],
        predicted_binary[..., channel],
    )

    print(f"{name}: {score:.4f}")

overall_dice = calculate_dice(
    ground_truth_binary,
    predicted_binary,
)

print(f"Overall Dice: {overall_dice:.4f}")


# ============================================================
# Visualise MRI and segmentation masks
# ============================================================

plt.style.use("ggplot")

fig, axes = plt.subplots(
    2,
    4,
    figsize=(18, 9),
)

fig.suptitle(
    f"MRI Segmentation Prediction: {target_filename}",
    fontsize=16,
)

# Display each MRI modality
modality_names = [
    "T1",
    "T1ce",
    "T2",
    "FLAIR",
]

for channel in range(min(image.shape[-1], 4)):
    axes[0, channel].imshow(
        image[..., channel],
        cmap="gray",
    )

    axes[0, channel].set_title(
        modality_names[channel]
    )

    axes[0, channel].axis("off")


# RGB mask visualisation: NEC=Red, ED=Green, ET=Blue
def mask_to_rgb(mask):
    mask = np.asarray(mask)

    rgb = np.zeros(
        (mask.shape[0], mask.shape[1], 3),
        dtype=np.float32,
    )

    rgb[..., 0] = mask[..., 0]  # NEC
    rgb[..., 1] = mask[..., 1]  # ED
    rgb[..., 2] = mask[..., 2]  # ET

    return np.clip(rgb, 0.0, 1.0)


axes[1, 0].imshow(
    image[..., 0],
    cmap="gray",
)

axes[1, 0].set_title("MRI (T1)")
axes[1, 0].axis("off")

axes[1, 1].imshow(
    mask_to_rgb(ground_truth_binary)
)

axes[1, 1].set_title(
    "Ground Truth\nRed=NEC, Green=ED, Blue=ET"
)

axes[1, 1].axis("off")

axes[1, 2].imshow(
    mask_to_rgb(predicted_binary)
)

axes[1, 2].set_title(
    "Predicted Mask\nThreshold=0.5"
)

axes[1, 2].axis("off")

axes[1, 3].imshow(
    mask_to_rgb(predicted_probabilities)
)

axes[1, 3].set_title(
    "Predicted Probabilities\nRed=NEC, Green=ED, Blue=ET"
)

axes[1, 3].axis("off")

plt.tight_layout()
plt.show()


# ============================================================
# Display individual ground-truth and predicted channels
# ============================================================

fig, axes = plt.subplots(
    3,
    3,
    figsize=(13, 12),
)

for channel, name in enumerate(channel_names):
    axes[channel, 0].imshow(
        ground_truth_binary[..., channel],
        cmap="gray",
        vmin=0,
        vmax=1,
    )

    axes[channel, 0].set_title(
        f"Actual {name}"
    )

    axes[channel, 1].imshow(
        predicted_binary[..., channel],
        cmap="gray",
        vmin=0,
        vmax=1,
    )

    axes[channel, 1].set_title(
        f"Predicted {name}"
    )

    axes[channel, 2].imshow(
        predicted_probabilities[..., channel],
        cmap="viridis",
        vmin=0,
        vmax=1,
    )

    axes[channel, 2].set_title(
        f"{name} Probability"
    )

    for ax in axes[channel]:
        ax.axis("off")

plt.suptitle(
    f"Per-channel Segmentation: {target_filename}",
    fontsize=16,
)

plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

In [5]:
import os
import shutil
import tempfile
import h5py
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf


# ============================================================
# Dice metric and loss functions
# ============================================================

def dice_coef(y_true, y_pred):
    smooth = 1e-6

    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    y_true = tf.reshape(y_true, [-1])
    y_pred = tf.reshape(y_pred, [-1])

    intersection = tf.reduce_sum(y_true * y_pred)

    return (
        (2.0 * intersection + smooth)
        / (
            tf.reduce_sum(y_true)
            + tf.reduce_sum(y_pred)
            + smooth
        )
    )


def binary_dice_coef(y_true, y_pred):
    y_pred = tf.cast(y_pred >= 0.5, tf.float32)
    return dice_coef(y_true, y_pred)


def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)


def combined_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(
        y_true,
        y_pred,
    )
    return tf.reduce_mean(bce) + dice_loss(y_true, y_pred)


# ============================================================
# Load saved model
# ============================================================

model_path = os.path.join(
    os.path.dirname(os.path.abspath("cnn_semantic.ipynb")),
    "unet_brain_model_best.keras",
)

if not os.path.isfile(model_path):
    raise FileNotFoundError(
        f"Model file not found: {model_path}"
    )

custom_objects = {
    "dice_coef": dice_coef,
    "binary_dice_coef": binary_dice_coef,
    "dice_loss": dice_loss,
    "combined_loss": combined_loss,
}

if h5py.is_hdf5(model_path):

    class CompatibleConv2DTranspose(
        tf.keras.layers.Conv2DTranspose
    ):
        @classmethod
        def from_config(cls, config):
            config = dict(config)
            groups = config.pop("groups", 1)

            if groups != 1:
                raise ValueError(
                    f"Cannot load Conv2DTranspose with groups={groups}"
                )

            return super().from_config(config)

    with tempfile.TemporaryDirectory() as temp_dir:
        legacy_model_path = os.path.join(
            temp_dir,
            "unet_brain_model_best.h5",
        )

        shutil.copy2(
            model_path,
            legacy_model_path,
        )

        brain_model = tf.keras.models.load_model(
            legacy_model_path,
            custom_objects={
                **custom_objects,
                "Conv2DTranspose": CompatibleConv2DTranspose,
            },
            compile=False,
        )

else:
    brain_model = tf.keras.models.load_model(
        model_path,
        custom_objects=custom_objects,
        compile=False,
    )

print("Model loaded successfully.")
print("Model input shape:", brain_model.input_shape)
print("Test data shape:", X_test.shape)


# ============================================================
# Generate predictions
# ============================================================

y_pred = brain_model.predict(
    X_test,
    verbose=0,
)

print("Prediction shape:", y_pred.shape)


# ============================================================
# Select random test images with non-empty masks
# ============================================================

valid_indices = np.flatnonzero(
    np.any(Y_test > 0, axis=(1, 2, 3))
)

if len(valid_indices) == 0:
    raise ValueError(
        "No test images contain a ground-truth mask."
    )

number_of_examples = min(
    500,
    len(valid_indices),
)

selected_indices = np.random.choice(
    valid_indices,
    size=number_of_examples,
    replace=False,
)

print(
    "Random test images with ground truth:",
    selected_indices,
)


# ============================================================
# Plot styling
# ============================================================

plt.style.use("ggplot")
plt.rcParams["figure.facecolor"] = "#171717"
plt.rcParams["text.color"] = "#AAAAAA"


# ============================================================
# Display mask channels as RGB
# ============================================================

def display_mask_channels_as_rgb(
    mask,
    title="Mask Channels as RGB",
):
    channel_names = [
        "Necrotic (NEC)",
        "Edema (ED)",
        "Tumour (ET)",
    ]

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(9.75, 5),
    )

    for idx, ax in enumerate(axes):
        rgb_mask = np.zeros(
            (mask.shape[1], mask.shape[2], 3),
            dtype=np.uint8,
        )

        rgb_mask[..., idx] = (
            np.clip(mask[idx, :, :], 0, 1) * 255
        ).astype(np.uint8)

        ax.imshow(rgb_mask)
        ax.axis("off")
        ax.set_title(channel_names[idx])

    plt.suptitle(
        title,
        fontsize=20,
        y=0.93,
    )

    plt.tight_layout()
    plt.show()


# ============================================================
# Calculate Dice, sensitivity and specificity
# ============================================================

def calculate_metrics(
    ground_truth,
    prediction,
    threshold=0.65,
):
    """
    Calculates metrics using binary segmentation masks.

    Dice = 2TP / (2TP + FP + FN)
    Sensitivity = TP / (TP + FN)
    Specificity = TN / (TN + FP)
    """

    ground_truth = np.asarray(
        ground_truth >= 0.5,
        dtype=bool,
    )

    prediction = np.asarray(
        prediction >= threshold,
        dtype=bool,
    )

    tp = np.sum(ground_truth & prediction)
    tn = np.sum(~ground_truth & ~prediction)
    fp = np.sum(~ground_truth & prediction)
    fn = np.sum(ground_truth & ~prediction)

    smooth = 1e-6

    dice = (
        (2.0 * tp + smooth)
        / (2.0 * tp + fp + fn + smooth)
    )

    sensitivity = (
        (tp + smooth)
        / (tp + fn + smooth)
    )

    specificity = (
        (tn + smooth)
        / (tn + fp + smooth)
    )

    return {
        "dice": float(dice),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
    }


# ============================================================
# Evaluate selected images
# ============================================================

dice_scores = []
sensitivity_scores = []
specificity_scores = []

channel_names = [
    "NEC",
    "ED",
    "ET",
]

channel_metrics = {
    name: {
        "dice": [],
        "sensitivity": [],
        "specificity": [],
    }
    for name in channel_names
}

for example_number, idx in enumerate(
    selected_indices,
    start=1,
):
    ground_truth = Y_test[idx]
    prediction = y_pred[idx]

    # Binary predictions for visualisation and evaluation
    binary_prediction = (
        prediction >= 0.5
    ).astype(np.float32)

    binary_ground_truth = (
        ground_truth >= 0.5
    ).astype(np.float32)

    # Overall metrics across all pixels and all channels
    overall = calculate_metrics(
        ground_truth,
        prediction,
    )

    dice_scores.append(overall["dice"])
    sensitivity_scores.append(overall["sensitivity"])
    specificity_scores.append(overall["specificity"])

    print(
        f"\nExample {example_number}: "
        f"image index {idx}"
    )

    print(
        f"  Overall Dice:        {overall['dice']:.4f}"
    )

    print(
        f"  Overall Sensitivity: {overall['sensitivity']:.4f}"
    )

    print(
        f"  Overall Specificity: {overall['specificity']:.4f}"
    )

    # Per-channel metrics
    for channel, name in enumerate(channel_names):
        metrics = calculate_metrics(
            ground_truth[..., channel],
            prediction[..., channel],
        )

        channel_metrics[name]["dice"].append(
            metrics["dice"]
        )

        channel_metrics[name]["sensitivity"].append(
            metrics["sensitivity"]
        )

        channel_metrics[name]["specificity"].append(
            metrics["specificity"]
        )

        print(
            f"  {name}: "
            f"Dice={metrics['dice']:.4f}, "
            f"Sensitivity={metrics['sensitivity']:.4f}, "
            f"Specificity={metrics['specificity']:.4f}"
        )

    # Display actual mask
    display_mask_channels_as_rgb(
        binary_ground_truth.transpose(2, 0, 1),
        title=f"Actual Mask - Example {example_number}",
    )

    # Display predicted binary mask
    display_mask_channels_as_rgb(
        binary_prediction.transpose(2, 0, 1),
        title=(
            f"Predicted Mask - Example {example_number} "
            f"(Dice: {overall['dice']:.4f})"
        ),
    )


# ============================================================
# Print average overall metrics
# ============================================================

print("\n" + "=" * 55)
print("AVERAGE EVALUATION METRICS")
print("=" * 55)

print(
    f"Average Dice:        "
    f"{np.mean(dice_scores):.4f}"
)

print(
    f"Average Sensitivity: "
    f"{np.mean(sensitivity_scores):.4f}"
)

print(
    f"Average Specificity: "
    f"{np.mean(specificity_scores):.4f}"
)


# ============================================================
# Print average per-channel metrics
# ============================================================

print("\n" + "=" * 55)
print("AVERAGE METRICS PER TUMOUR CHANNEL")
print("=" * 55)

for name in channel_names:
    print(f"\n{name}:")

    print(
        f"  Average Dice:        "
        f"{np.mean(channel_metrics[name]['dice']):.4f}"
    )

    print(
        f"  Average Sensitivity: "
        f"{np.mean(channel_metrics[name]['sensitivity']):.4f}"
    )

    print(
        f"  Average Specificity: "
        f"{np.mean(channel_metrics[name]['specificity']):.4f}"
    )

print("\nEvaluation completed.")

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:

import os
import shutil
import tempfile
import h5py
import numpy as np
import tensorflow as tf


# ============================================================
# Configuration
# ============================================================

MODEL_FILENAME = "unet_brain_model_best_sprint7.keras"
NUMBER_OF_SLICES = 589
THRESHOLD = 0.5
SEED = 14

CHANNEL_NAMES = ["NEC", "ED", "ET"]


# ============================================================
# Dice metric and loss functions
# ============================================================

def dice_coef(y_true, y_pred):
    smooth = 1e-6

    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    y_true = tf.reshape(y_true, [-1])
    y_pred = tf.reshape(y_pred, [-1])

    intersection = tf.reduce_sum(y_true * y_pred)

    return (
        (2.0 * intersection + smooth)
        / (
            tf.reduce_sum(y_true)
            + tf.reduce_sum(y_pred)
            + smooth
        )
    )


def binary_dice_coef(y_true, y_pred):
    y_pred = tf.cast(y_pred >= 0.5, tf.float32)
    return dice_coef(y_true, y_pred)


def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)


def combined_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(
        y_true,
        y_pred,
    )

    return tf.reduce_mean(bce) + dice_loss(
        y_true,
        y_pred,
    )


# ============================================================
# Load saved model
# ============================================================

model_path = os.path.join(
    os.path.dirname(os.path.abspath("cnn_semantic.ipynb")),
    MODEL_FILENAME,
)

if not os.path.isfile(model_path):
    raise FileNotFoundError(
        f"Model file not found: {model_path}"
    )

custom_objects = {
    "dice_coef": dice_coef,
    "binary_dice_coef": binary_dice_coef,
    "dice_loss": dice_loss,
    "combined_loss": combined_loss,
}

if h5py.is_hdf5(model_path):

    class CompatibleConv2DTranspose(
        tf.keras.layers.Conv2DTranspose
    ):
        @classmethod
        def from_config(cls, config):
            config = dict(config)
            groups = config.pop("groups", 1)

            if groups != 1:
                raise ValueError(
                    "Cannot load Conv2DTranspose "
                    f"with groups={groups}"
                )

            return super().from_config(config)

    with tempfile.TemporaryDirectory() as temp_dir:
        legacy_model_path = os.path.join(
            temp_dir,
            "unet_brain_model_best.h5",
        )

        shutil.copy2(
            model_path,
            legacy_model_path,
        )

        brain_model = tf.keras.models.load_model(
            legacy_model_path,
            custom_objects={
                **custom_objects,
                "Conv2DTranspose": CompatibleConv2DTranspose,
            },
            compile=False,
        )

else:
    brain_model = tf.keras.models.load_model(
        model_path,
        custom_objects=custom_objects,
        compile=False,
    )

print("Model loaded successfully.")
print("Model input shape:", brain_model.input_shape)
print("Test images:", X_test.shape)
print("Ground-truth masks:", Y_test.shape)


# ============================================================
# Validate test data
# ============================================================

if len(X_test) != len(Y_test):
    raise ValueError(
        "X_test and Y_test must contain the same number of slices."
    )

if Y_test.ndim != 4 or Y_test.shape[-1] != 3:
    raise ValueError(
        "Expected Y_test shape (N, H, W, 3), "
        f"but received {Y_test.shape}."
    )

if len(Y_test) < NUMBER_OF_SLICES:
    raise ValueError(
        f"Only {len(Y_test)} test slices are available. "
        f"Cannot evaluate {NUMBER_OF_SLICES} unique slices."
    )




# A slice qualifies if any ground-truth tumour channel
# contains at least one positive pixel.
valid_indices = np.flatnonzero(
    np.any(Y_test > 0.5, axis=(1, 2, 3))
)

print("Slices containing tumour:", len(valid_indices))

if len(valid_indices) < NUMBER_OF_SLICES:
    raise ValueError(
        f"Only {len(valid_indices)} slices contain tumour masks. "
        f"At least {NUMBER_OF_SLICES} are required. "
        "Use a larger test set or reduce NUMBER_OF_SLICES."
    )

rng = np.random.default_rng(SEED)

selected_indices = rng.choice(
    valid_indices,
    size=NUMBER_OF_SLICES,
    replace=False,
)

assert len(np.unique(selected_indices)) == NUMBER_OF_SLICES

print("Unique slices selected:", len(selected_indices))


# ============================================================
# Generate predictions for the selected slices only
# ============================================================

X_selected = X_test[selected_indices]
Y_selected = Y_test[selected_indices]

y_pred = brain_model.predict(
    X_selected,
    batch_size=16,
    verbose=0,
)

print("Prediction shape:", y_pred.shape)

if y_pred.shape != Y_selected.shape:
    raise ValueError(
        "Prediction and ground-truth shapes do not match: "
        f"{y_pred.shape} versus {Y_selected.shape}"
    )


# ============================================================
# Calculate Dice, sensitivity and specificity
# ============================================================

def calculate_metrics(
    ground_truth,
    prediction,
    threshold=THRESHOLD,
):
    """
    Calculate binary segmentation metrics.

    Dice       = 2*TP / (2*TP + FP + FN)
    Sensitivity = TP / (TP + FN)
    Specificity = TN / (TN + FP)

    Ground truth and predictions are evaluated pixel by pixel.
    """

    ground_truth = np.asarray(
        ground_truth >= 0.5,
        dtype=bool,
    )

    prediction = np.asarray(
        prediction >= threshold,
        dtype=bool,
    )

    tp = np.count_nonzero(
        ground_truth & prediction
    )

    tn = np.count_nonzero(
        ~ground_truth & ~prediction
    )

    fp = np.count_nonzero(
        ~ground_truth & prediction
    )

    fn = np.count_nonzero(
        ground_truth & ~prediction
    )

    smooth = 1e-6

    dice = (
        (2.0 * tp + smooth)
        / (2.0 * tp + fp + fn + smooth)
    )

    sensitivity = (
        (tp + smooth)
        / (tp + fn + smooth)
    )

    specificity = (
        (tn + smooth)
        / (tn + fp + smooth)
    )

    return {
        "dice": float(dice),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
    }


# ============================================================
# Evaluate each selected slice
# ============================================================

overall_metrics = {
    "dice": [],
    "sensitivity": [],
    "specificity": [],
}

channel_metrics = {
    name: {
        "dice": [],
        "sensitivity": [],
        "specificity": [],
    }
    for name in CHANNEL_NAMES
}

for i in range(NUMBER_OF_SLICES):

    ground_truth = Y_selected[i]
    prediction = y_pred[i]

    # Overall metrics across all pixels and all 3 channels
    overall = calculate_metrics(
        ground_truth,
        prediction,
    )

    for metric_name in overall_metrics:
        overall_metrics[metric_name].append(
            overall[metric_name]
        )

    # Individual tumour channel metrics
    for channel, name in enumerate(CHANNEL_NAMES):

        metrics = calculate_metrics(
            ground_truth[..., channel],
            prediction[..., channel],
        )

        for metric_name in channel_metrics[name]:
            channel_metrics[name][metric_name].append(
                metrics[metric_name]
            )


# ============================================================
# Print average overall metrics
# ============================================================

print("\n" + "=" * 55)
print("AVERAGE METRICS - 500 UNIQUE TUMOUR SLICES")
print("=" * 55)

print(f"Slices evaluated: {NUMBER_OF_SLICES}")
print(f"Dice:             {np.mean(overall_metrics['dice']):.6f}")
print(
    f"Sensitivity:      "
    f"{np.mean(overall_metrics['sensitivity']):.6f}"
)
print(
    f"Specificity:      "
    f"{np.mean(overall_metrics['specificity']):.6f}"
)


# ============================================================
# Print average metrics per tumour channel
# ============================================================

print("\n" + "=" * 55)
print("AVERAGE METRICS PER TUMOUR CHANNEL")
print("=" * 55)

for name in CHANNEL_NAMES:

    print(f"\n{name}")

    print(
        f"  Dice:        "
        f"{np.mean(channel_metrics[name]['dice']):.6f}"
    )

    print(
        f"  Sensitivity: "
        f"{np.mean(channel_metrics[name]['sensitivity']):.6f}"
    )

    print(
        f"  Specificity: "
        f"{np.mean(channel_metrics[name]['specificity']):.6f}"
    )


# ============================================================
# Final verification
# ============================================================

print("\n" + "=" * 55)
print("EVALUATION COMPLETED")
print("=" * 55)

print("Unique slices evaluated:", len(selected_indices))
print("Duplicate slice indices:", (
    len(selected_indices) - len(np.unique(selected_indices))
))
print("Images plotted: 0")


Model loaded successfully.
Model input shape: (None, 240, 240, 4)
Test images: (800, 240, 240, 4)
Ground-truth masks: (800, 240, 240, 3)
Slices containing tumour: 350


ValueError: Only 350 slices contain tumour masks. At least 589 are required. Use a larger test set or reduce NUMBER_OF_SLICES.

In [11]:

import os
import re
import random
import h5py
import numpy as np
import tensorflow as tf
from collections import defaultdict

# ============================================================
# CONFIGURATION
# ============================================================

MODEL_PATH = (
    r"C:\Users\Alex\Music\ComputingTechnologyProjectBP45"
    r"\benchmark\unet_brain_model_best_sprint7.keras"
)

# Change this to your H5 dataset directory if `directory`
# has not already been defined in your notebook.
DATA_DIRECTORY = None

NUM_VOLUMES = 50
SEED = 5
THRESHOLD = 0.3
EPSILON = 1e-7

# Set this to True only if your model was trained with this
# same per-slice preprocessing.
APPLY_PREPROCESSING = True

# ============================================================
# DATA DIRECTORY
# ============================================================

if DATA_DIRECTORY is not None:
    directory = DATA_DIRECTORY
elif "directory" not in globals():
    raise NameError(
        "Set DATA_DIRECTORY to the folder containing your H5 files."
    )

directory = os.path.abspath(directory)

if not os.path.isdir(directory):
    raise FileNotFoundError(
        f"Dataset directory does not exist: {directory}"
    )

# ============================================================
# CUSTOM METRICS / LOSS FOR MODEL LOADING
# ============================================================

def dice_coef(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    intersection = tf.reduce_sum(
        y_true * y_pred, axis=[1, 2, 3]
    )

    denominator = (
        tf.reduce_sum(y_true, axis=[1, 2, 3])
        + tf.reduce_sum(y_pred, axis=[1, 2, 3])
    )

    dice = (2.0 * intersection + EPSILON) / (
        denominator + EPSILON
    )

    return tf.reduce_mean(dice)


def binary_dice_coef(y_true, y_pred):
    y_true = tf.cast(y_true > 0.5, tf.float32)
    y_pred = tf.cast(y_pred > 0.5, tf.float32)

    intersection = tf.reduce_sum(
        y_true * y_pred, axis=[1, 2, 3]
    )

    denominator = (
        tf.reduce_sum(y_true, axis=[1, 2, 3])
        + tf.reduce_sum(y_pred, axis=[1, 2, 3])
    )

    dice = (2.0 * intersection + EPSILON) / (
        denominator + EPSILON
    )

    return tf.reduce_mean(dice)


def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)


def combined_loss(y_true, y_pred):
    bce = tf.reduce_mean(
        tf.keras.losses.binary_crossentropy(
            y_true, y_pred
        )
    )
    return bce + dice_loss(y_true, y_pred)


# ============================================================
# LOAD MODEL
# ============================================================

if not os.path.isfile(MODEL_PATH):
    raise FileNotFoundError(
        f"Model file not found: {MODEL_PATH}"
    )

brain_model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={
        "dice_coef": dice_coef,
        "binary_dice_coef": binary_dice_coef,
        "dice_loss": dice_loss,
        "combined_loss": combined_loss,
    },
    compile=False,
)

print("Model loaded successfully.")
print("Model input shape:", brain_model.input_shape)
print("Model output shape:", brain_model.output_shape)

# ============================================================
# PREPROCESSING
# Must match the preprocessing used during training.
# ============================================================

def preprocess_mri_image(image):
    image = np.asarray(image, dtype=np.float32)

    if image.ndim != 3:
        raise ValueError(
            f"Expected H x W x C image, got {image.shape}"
        )

    image = np.nan_to_num(
        image,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    processed = np.zeros_like(image, dtype=np.float32)
    brain_mask = np.any(image != 0, axis=-1)

    if not np.any(brain_mask):
        return processed

    for channel in range(image.shape[-1]):
        channel_data = image[..., channel]
        brain_values = channel_data[brain_mask]
        brain_values = brain_values[np.isfinite(brain_values)]

        if brain_values.size == 0:
            continue

        low = np.percentile(brain_values, 0.5)
        high = np.percentile(brain_values, 99.5)

        clipped = np.clip(channel_data, low, high)
        valid_values = clipped[brain_mask]

        mean = np.mean(valid_values)
        std = np.std(valid_values)

        if std > 1e-6:
            processed[..., channel] = (
                clipped - mean
            ) / std
        else:
            processed[..., channel] = clipped - mean

        processed[..., channel][~brain_mask] = 0.0

    return processed.astype(np.float32)


# ============================================================
# FIND AND GROUP FILES BY VOLUME
# Supports:
# volume_1_slice_0.h5
# vol-1-slice0.h5
# ============================================================

patterns = [
    re.compile(
        r"^volume_(\d+)_slice_(\d+)\.h5$",
        re.IGNORECASE
    ),
    re.compile(
        r"^vol-(\d+)-slice(\d+)\.h5$",
        re.IGNORECASE
    ),
]

volume_files = defaultdict(list)

for filename in os.listdir(directory):
    if not filename.lower().endswith(".h5"):
        continue

    for pattern in patterns:
        match = pattern.match(filename)

        if match:
            volume_id = int(match.group(1))
            slice_id = int(match.group(2))

            volume_files[volume_id].append(
                (slice_id, os.path.join(directory, filename))
            )
            break

if not volume_files:
    raise RuntimeError(
        "No matching H5 files found. Check the filename format "
        "and DATA_DIRECTORY."
    )

for volume_id in volume_files:
    volume_files[volume_id].sort(key=lambda item: item[0])

available_volume_ids = sorted(volume_files.keys())

print(f"\nVolumes discovered: {len(available_volume_ids)}")

if len(available_volume_ids) < NUM_VOLUMES:
    raise ValueError(
        f"Only {len(available_volume_ids)} volumes were found, "
        f"but {NUM_VOLUMES} were requested."
    )

# ============================================================
# SELECT 50 DIFFERENT VOLUMES
# ============================================================

rng = random.Random(SEED)

selected_volume_ids = rng.sample(
    available_volume_ids,
    NUM_VOLUMES
)

selected_volume_ids.sort()

print(f"Volumes selected for evaluation: {NUM_VOLUMES}")

# ============================================================
# METRIC CALCULATION
# ============================================================

def calculate_binary_metrics(y_true, y_pred):
    """
    Calculates metrics for one binary mask.

    y_true and y_pred must already be binary arrays.
    """

    y_true = np.asarray(y_true, dtype=bool)
    y_pred = np.asarray(y_pred, dtype=bool)

    tp = int(np.sum(y_true & y_pred))
    fp = int(np.sum(~y_true & y_pred))
    fn = int(np.sum(y_true & ~y_pred))
    tn = int(np.sum(~y_true & ~y_pred))

    dice = (2 * tp) / (2 * tp + fp + fn + EPSILON)
    precision = tp / (tp + fp + EPSILON)
    sensitivity = tp / (tp + fn + EPSILON)
    specificity = tn / (tn + fp + EPSILON)
    iou = tp / (tp + fp + fn + EPSILON)

    return {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "Dice": float(dice),
        "Precision": float(precision),
        "Sensitivity": float(sensitivity),
        "Specificity": float(specificity),
        "IoU": float(iou),
    }


# ============================================================
# EVALUATE ONE TUMOUR-CONTAINING SLICE PER VOLUME
# ============================================================

channel_names = ["NEC", "ED", "ET"]

volume_results = []
skipped_volumes = []

for volume_number, volume_id in enumerate(
    selected_volume_ids, start=1
):
    print(
        f"\n[{volume_number}/{NUM_VOLUMES}] "
        f"Evaluating volume {volume_id}"
    )

    # Shuffle slices so we do not always select the first
    # tumour-containing slice in the volume.
    candidate_slices = volume_files[volume_id].copy()
    rng.shuffle(candidate_slices)

    selected_image = None
    selected_mask = None
    selected_filename = None

    # Find one slice containing at least one positive tumour pixel.
    for slice_id, filepath in candidate_slices:
        try:
            with h5py.File(filepath, "r") as h5_file:
                image = np.asarray(
                    h5_file["image"],
                    dtype=np.float32
                )

                mask = np.asarray(
                    h5_file["mask"],
                    dtype=np.float32
                )

        except Exception as exc:
            print(f"Could not read {filepath}: {exc}")
            continue

        if mask.ndim != 3 or mask.shape[-1] != 3:
            print(
                f"Skipping unexpected mask shape "
                f"{mask.shape}: {os.path.basename(filepath)}"
            )
            continue

        if image.ndim != 3:
            print(
                f"Skipping unexpected image shape "
                f"{image.shape}: {os.path.basename(filepath)}"
            )
            continue

        # A valid tumour slice must have at least one positive
        # pixel in any of the three mask channels.
        if np.any(mask > THRESHOLD):
            selected_image = image
            selected_mask = mask
            selected_filename = os.path.basename(filepath)
            break

    if selected_image is None:
        print(
            f"WARNING: No tumour-containing slice found "
            f"for volume {volume_id}. Skipping it."
        )
        skipped_volumes.append(volume_id)
        continue

    # Preprocess image if this matches the training pipeline.
    if APPLY_PREPROCESSING:
        selected_image = preprocess_mri_image(selected_image)
    else:
        selected_image = np.nan_to_num(
            selected_image,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        ).astype(np.float32)

    # Prepare batch.
    X = np.expand_dims(selected_image, axis=0)
    Y_true = np.asarray(selected_mask, dtype=np.float32)

    # Predict tumour mask.
    prediction = brain_model.predict(X, verbose=0)

    # Some models return a list of outputs.
    if isinstance(prediction, (list, tuple)):
        prediction = prediction[0]

    Y_pred = np.asarray(prediction[0], dtype=np.float32)

    if Y_pred.shape != Y_true.shape:
        raise ValueError(
            f"Shape mismatch for {selected_filename}: "
            f"ground truth {Y_true.shape}, prediction {Y_pred.shape}"
        )

    # Convert probabilities into binary masks.
    Y_true_binary = Y_true > THRESHOLD
    Y_pred_binary = Y_pred > THRESHOLD

    # Overall metrics: combine all pixels and all 3 channels.
    overall_metrics = calculate_binary_metrics(
        Y_true_binary,
        Y_pred_binary
    )

    # Per-channel metrics.
    per_channel_metrics = {}

    for channel_index, channel_name in enumerate(channel_names):
        per_channel_metrics[channel_name] = (
            calculate_binary_metrics(
                Y_true_binary[..., channel_index],
                Y_pred_binary[..., channel_index]
            )
        )

    volume_results.append({
        "volume_id": volume_id,
        "filename": selected_filename,
        "overall": overall_metrics,
        "channels": per_channel_metrics,
    })

    print(f"Selected slice: {selected_filename}")
    print(
        f"Overall Dice:      "
        f"{overall_metrics['Dice']:.4f}"
    )
    print(
        f"Overall Precision: "
        f"{overall_metrics['Precision']:.4f}"
    )
    print(
        f"Overall Sensitivity:"
        f" {overall_metrics['Sensitivity']:.4f}"
    )
    print(
        f"Overall Specificity:"
        f" {overall_metrics['Specificity']:.4f}"
    )
    print(
        f"Overall IoU:        "
        f"{overall_metrics['IoU']:.4f}"
    )

    for channel_name in channel_names:
        metrics = per_channel_metrics[channel_name]

        print(
            f"  {channel_name}: "
            f"Dice={metrics['Dice']:.4f}, "
            f"Precision={metrics['Precision']:.4f}, "
            f"Sensitivity={metrics['Sensitivity']:.4f}, "
            f"Specificity={metrics['Specificity']:.4f}, "
            f"IoU={metrics['IoU']:.4f}"
        )


# ============================================================
# AVERAGE METRICS ACROSS EVALUATED VOLUMES
# ============================================================

if not volume_results:
    raise RuntimeError(
        "No volumes could be evaluated. Check your filenames, "
        "mask values, preprocessing, and dataset directory."
    )

metric_names = [
    "Dice",
    "Precision",
    "Sensitivity",
    "Specificity",
    "IoU",
]

evaluated_count = len(volume_results)

print("\n" + "=" * 65)
print("AVERAGE METRICS ACROSS VOLUMES")
print("=" * 65)

print(f"Requested volumes: {NUM_VOLUMES}")
print(f"Successfully evaluated: {evaluated_count}")
print(f"Skipped volumes: {len(skipped_volumes)}")

# Average each volume's overall metrics equally.
# This is a macro-average across the evaluated volumes.
print("\nOVERALL (all tumour channels combined)")
for metric_name in metric_names:
    values = [
        result["overall"][metric_name]
        for result in volume_results
    ]

    print(
        f"Average {metric_name:<12}: "
        f"{np.mean(values):.4f} "
        f"({np.mean(values) * 100:.2f}%)"
    )

# Average the metrics for each tumour channel separately.
for channel_name in channel_names:
    print(f"\n{channel_name}")

    for metric_name in metric_names:
        values = [
            result["channels"][channel_name][metric_name]
            for result in volume_results
        ]

        print(
            f"Average {metric_name:<12}: "
            f"{np.mean(values):.4f} "
            f"({np.mean(values) * 100:.2f}%)"
        )

# ============================================================
# SAVE RESULTS TO CSV
# ============================================================

import csv

output_csv = os.path.join(
    directory,
    "average_metrics_50_volumes.csv"
)

csv_columns = [
    "volume_id",
    "filename",
    "overall_Dice",
    "overall_Precision",
    "overall_Sensitivity",
    "overall_Specificity",
    "overall_IoU",
]

for channel_name in channel_names:
    for metric_name in metric_names:
        csv_columns.append(
            f"{channel_name}_{metric_name}"
        )

with open(output_csv, "w", newline="", encoding="utf-8") as csv_file:
    writer = csv.DictWriter(
        csv_file,
        fieldnames=csv_columns
    )
    writer.writeheader()

    for result in volume_results:
        row = {
            "volume_id": result["volume_id"],
            "filename": result["filename"],
        }

        for metric_name in metric_names:
            row[f"overall_{metric_name}"] = (
                result["overall"][metric_name]
            )

        for channel_name in channel_names:
            for metric_name in metric_names:
                row[f"{channel_name}_{metric_name}"] = (
                    result["channels"][channel_name][metric_name]
                )

        writer.writerow(row)

print(f"\nPer-volume results saved to:\n{output_csv}")

if skipped_volumes:
    print("\nSkipped volume IDs:", skipped_volumes)

print("\nEvaluation complete.")


Model loaded successfully.
Model input shape: (None, 240, 240, 4)
Model output shape: (None, 240, 240, 3)

Volumes discovered: 369
Volumes selected for evaluation: 50

[1/50] Evaluating volume 1
Selected slice: volume_1_slice_49.h5
Overall Dice:      0.8579
Overall Precision: 0.8639
Overall Sensitivity: 0.8521
Overall Specificity: 0.9984
Overall IoU:        0.7512
  NEC: Dice=0.9179, Precision=0.9314, Sensitivity=0.9048, Specificity=0.9996, IoU=0.8482
  ED: Dice=0.8436, Precision=0.9000, Sensitivity=0.7939, Specificity=0.9983, IoU=0.7295
  ET: Dice=0.8519, Precision=0.7791, Sensitivity=0.9398, Specificity=0.9974, IoU=0.7421

[2/50] Evaluating volume 3
Selected slice: volume_3_slice_105.h5
Overall Dice:      0.8342
Overall Precision: 0.7685
Overall Sensitivity: 0.9121
Overall Specificity: 0.9996
Overall IoU:        0.7155
  NEC: Dice=0.0000, Precision=0.0000, Sensitivity=0.0000, Specificity=1.0000, IoU=0.0000
  ED: Dice=0.8342, Precision=0.7685, Sensitivity=0.9121, Specificity=0.9987, I

In [8]:

import os
import shutil
import tempfile
import zipfile
import h5py
import numpy as np
import tensorflow as tf


# ============================================================
# Configuration
# ============================================================

MODEL_FILENAME = "unet_brain_model_best_sprint7.keras"

# Volume to evaluate
VOLUME_ID = 1

# Random seed used to choose a tumour-containing slice
SEED = 14

THRESHOLD = 0.7

CHANNEL_NAMES = ["NEC", "ED", "ET"]


# ============================================================
# Dice metric and loss functions
# ============================================================

def dice_coef(y_true, y_pred):
    smooth = 1e-6

    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    y_true = tf.reshape(y_true, [-1])
    y_pred = tf.reshape(y_pred, [-1])

    intersection = tf.reduce_sum(y_true * y_pred)

    return (
        (2.0 * intersection + smooth)
        / (
            tf.reduce_sum(y_true)
            + tf.reduce_sum(y_pred)
            + smooth
        )
    )


def binary_dice_coef(y_true, y_pred):
    y_pred = tf.cast(y_pred >= 0.5, tf.float32)
    return dice_coef(y_true, y_pred)


def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)


def combined_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(
        y_true,
        y_pred,
    )

    return tf.reduce_mean(bce) + dice_loss(
        y_true,
        y_pred,
    )


# ============================================================
# Load saved model
# ============================================================

model_path = os.path.join(
    os.path.dirname(os.path.abspath("cnn_semantic.ipynb")),
    MODEL_FILENAME,
)

if not os.path.isfile(model_path):
    raise FileNotFoundError(
        f"Model file not found: {model_path}"
    )


custom_objects = {
    "dice_coef": dice_coef,
    "binary_dice_coef": binary_dice_coef,
    "dice_loss": dice_loss,
    "combined_loss": combined_loss,
}


if h5py.is_hdf5(model_path):

    class CompatibleConv2DTranspose(
        tf.keras.layers.Conv2DTranspose
    ):
        @classmethod
        def from_config(cls, config):
            config = dict(config)
            groups = config.pop("groups", 1)

            if groups != 1:
                raise ValueError(
                    "Cannot load Conv2DTranspose "
                    f"with groups={groups}"
                )

            return super().from_config(config)

    with tempfile.TemporaryDirectory() as temp_dir:

        legacy_model_path = os.path.join(
            temp_dir,
            "unet_brain_model_best.h5",
        )

        shutil.copy2(
            model_path,
            legacy_model_path,
        )

        brain_model = tf.keras.models.load_model(
            legacy_model_path,
            custom_objects={
                **custom_objects,
                "Conv2DTranspose": CompatibleConv2DTranspose,
            },
            compile=False,
        )

else:
    # Supports modern .keras archives when the installed
    # TensorFlow/Keras version is compatible with the model.
    brain_model = tf.keras.models.load_model(
        model_path,
        custom_objects=custom_objects,
        compile=False,
    )


print("Model loaded successfully.")
print("Model input shape:", brain_model.input_shape)
print("Model output shape:", brain_model.output_shape)


# ============================================================
# Find the data directory
# ============================================================
# If you already have a variable named directory, it will be used.
# Otherwise, set DATA_DIRECTORY to your H5 data folder.

DATA_DIRECTORY = None

if DATA_DIRECTORY is not None:
    directory = DATA_DIRECTORY

elif "directory" not in globals():
    raise NameError(
        "Set DATA_DIRECTORY to your H5 data folder, "
        "or define the variable directory before running this code."
    )

if not os.path.isdir(directory):
    raise NotADirectoryError(
        f"Data directory does not exist: {directory}"
    )

print("Data directory:", directory)


# ============================================================
# Find all slices belonging to the requested volume
# ============================================================

import re

filename_pattern = re.compile(
    rf"^volume_{VOLUME_ID}_slice_(\d+)\.h5$"
)

volume_slices = []

for filename in os.listdir(directory):

    match = filename_pattern.match(filename)

    if match:
        slice_id = int(match.group(1))

        volume_slices.append(
            (slice_id, filename)
        )

volume_slices.sort(key=lambda item: item[0])

if not volume_slices:
    raise FileNotFoundError(
        f"No H5 slices found for Volume {VOLUME_ID} "
        f"in {directory}. Expected filenames like "
        f"volume_{VOLUME_ID}_slice_0.h5"
    )

print(
    f"Total slices found in Volume {VOLUME_ID}: "
    f"{len(volume_slices)}"
)


# ============================================================
# Select a slice that contains actual tumour pixels
# ============================================================

rng = np.random.default_rng(SEED)

# Shuffle the search order so that the selected slice
# is random, rather than always the first qualifying slice.
search_order = rng.permutation(len(volume_slices))

selected_slice = None
image = None
mask = None

print("\nSearching for a slice with tumour pixels...")

for position in search_order:

    slice_id, filename = volume_slices[position]

    slice_path = os.path.join(
        directory,
        filename,
    )

    with h5py.File(slice_path, "r") as h5_file:

        current_image = np.asarray(
            h5_file["image"],
            dtype=np.float32,
        )

        current_mask = np.asarray(
            h5_file["mask"],
            dtype=np.float32,
        )

    if current_image.ndim != 3:
        raise ValueError(
            f"Unexpected image shape in {filename}: "
            f"{current_image.shape}"
        )

    if current_mask.ndim != 3 or current_mask.shape[-1] != 3:
        raise ValueError(
            f"Expected mask shape (H, W, 3) in {filename}, "
            f"got {current_mask.shape}"
        )

    if current_image.shape[:2] != current_mask.shape[:2]:
        raise ValueError(
            f"Image and mask dimensions do not match in {filename}"
        )

    # Count positive tumour pixels in each channel.
    tumour_pixel_counts = np.count_nonzero(
        current_mask > 0.5,
        axis=(0, 1),
    )

    total_tumour_pixels = int(
        np.sum(tumour_pixel_counts)
    )

    # Accept only slices with at least one positive tumour pixel.
    if total_tumour_pixels > 0:

        selected_slice = filename
        image = current_image
        mask = current_mask

        print("\nTumour-containing slice found.")
        print("Volume:", VOLUME_ID)
        print("Slice:", slice_id)
        print("Filename:", selected_slice)
        print(
            "NEC tumour pixels:",
            int(tumour_pixel_counts[0]),
        )
        print(
            "ED tumour pixels:",
            int(tumour_pixel_counts[1]),
        )
        print(
            "ET tumour pixels:",
            int(tumour_pixel_counts[2]),
        )
        print(
            "Total positive channel pixels:",
            total_tumour_pixels,
        )

        break


if selected_slice is None:
    raise ValueError(
        f"No tumour-containing slices were found in Volume "
        f"{VOLUME_ID}. Every mask was checked, and none had "
        f"pixels greater than 0.5."
    )


# ============================================================
# Prepare selected image and ground-truth mask
# ============================================================

X_selected = np.expand_dims(
    image,
    axis=0,
)

Y_selected = np.expand_dims(
    mask,
    axis=0,
)

print("\nSelected image shape:", X_selected.shape)
print("Selected mask shape:", Y_selected.shape)

assert np.any(Y_selected > 0.5), (
    "Selected mask unexpectedly contains no tumour pixels."
)


# ============================================================
# Generate prediction
# ============================================================

y_pred = brain_model.predict(
    X_selected,
    batch_size=1,
    verbose=0,
)

print("Prediction shape:", y_pred.shape)

if y_pred.shape != Y_selected.shape:
    raise ValueError(
        "Prediction and ground-truth shapes do not match: "
        f"{y_pred.shape} versus {Y_selected.shape}"
    )


# ============================================================
# Calculate Dice, sensitivity and specificity
# ============================================================

def calculate_metrics(
    ground_truth,
    prediction,
    threshold=THRESHOLD,
):

    ground_truth = np.asarray(
        ground_truth >= 0.5,
        dtype=bool,
    )

    prediction = np.asarray(
        prediction >= threshold,
        dtype=bool,
    )

    tp = np.count_nonzero(
        ground_truth & prediction
    )

    tn = np.count_nonzero(
        ~ground_truth & ~prediction
    )

    fp = np.count_nonzero(
        ~ground_truth & prediction
    )

    fn = np.count_nonzero(
        ground_truth & ~prediction
    )

    smooth = 1e-6

    dice = (
        (2.0 * tp + smooth)
        / (2.0 * tp + fp + fn + smooth)
    )

    sensitivity = (
        (tp + smooth)
        / (tp + fn + smooth)
    )

    specificity = (
        (tn + smooth)
        / (tn + fp + smooth)
    )

    return {
        "dice": float(dice),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
    }


# ============================================================
# Evaluate overall metrics
# ============================================================

ground_truth = Y_selected[0]
prediction = y_pred[0]

overall_metrics = calculate_metrics(
    ground_truth,
    prediction,
)


# ============================================================
# Evaluate individual tumour channels
# ============================================================

channel_metrics = {}

for channel, name in enumerate(CHANNEL_NAMES):

    channel_metrics[name] = calculate_metrics(
        ground_truth[..., channel],
        prediction[..., channel],
    )


# ============================================================
# Print overall results
# ============================================================

print("\n" + "=" * 55)
print("SEGMENTATION RESULTS")
print("=" * 55)

print("Volume:", VOLUME_ID)
print("Selected slice:", selected_slice)
print("Slices evaluated: 1")
print(
    "Ground-truth tumour pixels:",
    int(np.count_nonzero(ground_truth > 0.5)),
)

print(
    f"\nOverall Dice:        "
    f"{overall_metrics['dice']:.6f}"
)

print(
    f"Overall Sensitivity: "
    f"{overall_metrics['sensitivity']:.6f}"
)

print(
    f"Overall Specificity: "
    f"{overall_metrics['specificity']:.6f}"
)


# ============================================================
# Print channel results
# ============================================================

print("\n" + "=" * 55)
print("METRICS PER TUMOUR CHANNEL")
print("=" * 55)

for name in CHANNEL_NAMES:

    metrics = channel_metrics[name]

    print(f"\n{name}")

    print(f"  Dice:        {metrics['dice']:.6f}")
    print(f"  Sensitivity: {metrics['sensitivity']:.6f}")
    print(f"  Specificity: {metrics['specificity']:.6f}")

    print(f"  TP: {metrics['TP']}")
    print(f"  TN: {metrics['TN']}")
    print(f"  FP: {metrics['FP']}")
    print(f"  FN: {metrics['FN']}")


# ============================================================
# Final verification
# ============================================================

print("\n" + "=" * 55)
print("EVALUATION COMPLETED")
print("=" * 55)

print("Volume evaluated:", VOLUME_ID)
print("Slice evaluated:", selected_slice)
print("Ground-truth tumour present: Yes")
print("Images plotted: 0")


Model loaded successfully.
Model input shape: (None, 240, 240, 4)
Model output shape: (None, 240, 240, 3)
Data directory: C:\Users\Alex\Music\ComputingTechnologyProjectBP45\benchmark\content\data
Total slices found in Volume 1: 155

Searching for a slice with tumour pixels...

Tumour-containing slice found.
Volume: 1
Slice: 59
Filename: volume_1_slice_59.h5
NEC tumour pixels: 708
ED tumour pixels: 1929
ET tumour pixels: 1007
Total positive channel pixels: 3644

Selected image shape: (1, 240, 240, 4)
Selected mask shape: (1, 240, 240, 3)
Prediction shape: (1, 240, 240, 3)

SEGMENTATION RESULTS
Volume: 1
Selected slice: volume_1_slice_59.h5
Slices evaluated: 1
Ground-truth tumour pixels: 3644

Overall Dice:        0.836063
Overall Sensitivity: 0.763447
Overall Specificity: 0.998646

METRICS PER TUMOUR CHANNEL

NEC
  Dice:        0.905068
  Sensitivity: 0.895480
  Specificity: 0.998963
  TP: 634
  TN: 56833
  FP: 59
  FN: 74

ED
  Dice:        0.763941
  Sensitivity: 0.639191
  Specificit